# Kaggle raw/L2 dense diagnostics

This notebook runs the raw-vs-L2 patch diagnostic suite on Kaggle. It does not mount Google Drive. Checkpoints and ImageNet-100 images are read from `/kaggle/input`, and outputs are written to `/kaggle/working` so Kaggle commits them as notebook output.

Default mode is the full raw/L2 structural sweep over every checkpoint found in the attached Kaggle input dataset. For a quick smoke test, temporarily set `CHECKPOINT_EPOCH_FILTER = [215]`, `NUM_DSE_IMAGES = 128`, `NUM_VIS_IMAGES = 3`, `PATCH_DSE_GROUP_STRIDE = 8`, and `OUTPUT_RUN_SUFFIX = 'raw_l2_smoke'` in the configuration cell.


## 1. Clone the raw/L2 diagnostics branch

Kaggle notebook Internet must be enabled for this cell. If Internet is disabled, attach this GitHub repository as a Kaggle notebook source instead and set `REPO_DIR` to that local source path.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'dino'
BRANCH = 'codex/raw-l2-diagnostics'
REPO_URL = 'https://github.com/xbz123/dino-dense-degradation.git'

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

subprocess.run(['grep', '-R', 'raw_dse', '-n', str(REPO_DIR / 'analyze_patch_statistics.py')], check=True)
subprocess.run(['grep', '-R', 'fig_raw_vs_l2_dse', '-n', str(REPO_DIR / 'plot_dense_diagnostics.py')], check=True)

sys.path.insert(0, str(REPO_DIR))


## 2. Configure Kaggle input paths and evaluation settings

Attach the checkpoint dataset and ImageNet-100 dataset in the Kaggle notebook sidebar. The candidate lists below include both common Kaggle paths and the paths used in previous runs. If your dataset slug differs, add it to the candidate list.


In [ ]:
from pathlib import Path

CHECKPOINT_INPUT_CANDIDATES = [
    '/kaggle/input/dinocheckpoint',
    '/kaggle/input/dino-ckp',
    '/kaggle/input/datasets/bingzhouxie1/dino-ckp',
    '/kaggle/input/dino-train-v2/dino_output',
    '/kaggle/input/notebooks/bingzhouxie1/dino-train-v2/dino_output',
    '/kaggle/input/notebooks/bingzhouxie1/notebook342eaf8bd6/dino_output',
]

DSE_IMAGE_ROOT_CANDIDATES = [
    '/kaggle/input/imagenet100/ImageNet100/train',
    '/kaggle/input/imagenet100/train',
    '/kaggle/input/datasets/wilyzh/imagenet100/ImageNet100/train',
    '/kaggle/input/datasets/wilyzh/imagenet100/train',
]

# Full run by default. For a smoke test, use [215].
CHECKPOINT_EPOCH_FILTER = None

OUTPUT_ROOT = Path('/kaggle/working/dino_dense_degradation_eval')
OUTPUT_RUN_SUFFIX = 'raw_l2_full'
WORK_CKPT_DIR = Path('/kaggle/working/dino_eval_checkpoints')

# Kaggle raw/L2 run is patch diagnostics only by default. VOC JSON is optional.
RUN_VOC_EVAL = False

NUM_DSE_IMAGES = 2048
NUM_VIS_IMAGES = 6
PATCH_DIAG_BATCH_SIZE = 32
PATCH_DIAG_NUM_WORKERS = 2
CHECKPOINT_KEY = 'teacher'
MAX_SPECTRUM_TOKENS = 30000
MAX_KMEANS_TOKENS = 12000
PATCH_DSE_GROUP_STRIDE = 1
STRICT_INTERNAL_EPOCH = False

def first_existing_dir(candidates, label):
    for candidate in candidates:
        path = Path(candidate)
        if path.is_dir():
            return path
    print('Available /kaggle/input entries:')
    input_root = Path('/kaggle/input')
    if input_root.is_dir():
        for child in sorted(input_root.iterdir()):
            print(' -', child)
    raise FileNotFoundError(f'Could not find {label}. Checked: {candidates}')

CHECKPOINT_INPUT_DIR = first_existing_dir(CHECKPOINT_INPUT_CANDIDATES, 'checkpoint input directory')
DSE_IMAGE_ROOT = first_existing_dir(DSE_IMAGE_ROOT_CANDIDATES, 'ImageNet-100 train directory')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print('checkpoint input:', CHECKPOINT_INPUT_DIR)
print('checkpoint filter:', CHECKPOINT_EPOCH_FILTER)
print('DSE image root:', DSE_IMAGE_ROOT)
print('output root:', OUTPUT_ROOT)
print('output suffix:', OUTPUT_RUN_SUFFIX)
print('run VOC eval:', RUN_VOC_EVAL)


## 3. Discover and normalize checkpoints

This scans the selected Kaggle input folder for `checkpoint*.pth`, then copies recognized files to `/kaggle/working/dino_eval_checkpoints/checkpoint####.pth`.


In [ ]:
import json
import os
import shutil
from dense_eval_utils import build_run_output_root, discover_checkpoint_files

print('=== checkpoint input files ===')
for path in sorted(CHECKPOINT_INPUT_DIR.iterdir()):
    print(path.name)

selected = discover_checkpoint_files(CHECKPOINT_INPUT_DIR, epoch_filter=CHECKPOINT_EPOCH_FILTER)
assert selected, f'No recognizable checkpoint*.pth files found in {CHECKPOINT_INPUT_DIR}'

for path in WORK_CKPT_DIR.glob('checkpoint*.pth'):
    path.unlink()

prepared = []
print('=== selected checkpoints ===')
for item in selected:
    dst = WORK_CKPT_DIR / f'checkpoint{item.epoch:04d}.pth'
    shutil.copy2(item.path, dst)
    print(f'{item.epoch:>4}: {item.path} -> {dst} | internal_epoch={item.internal_epoch} | {item.size_mb:.1f} MB')
    prepared.append({
        'epoch': item.epoch,
        'source': str(item.path),
        'normalized': str(dst),
        'internal_epoch': item.internal_epoch,
        'size_mb': item.size_mb,
    })

(WORK_CKPT_DIR / 'selected_checkpoints.json').write_text(json.dumps(prepared, indent=2))

SELECTED_EPOCHS = [item['epoch'] for item in prepared]
FINAL_EPOCH = max(SELECTED_EPOCHS)
RUN_OUTPUT_ROOT = build_run_output_root(OUTPUT_ROOT, SELECTED_EPOCHS, suffix=OUTPUT_RUN_SUFFIX)
RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(WORK_CKPT_DIR / 'selected_checkpoints.json', RUN_OUTPUT_ROOT / 'selected_checkpoints.json')

VOC_JSON_CANDIDATES = [
    RUN_OUTPUT_ROOT / 'voc_all_checkpoints' / 'voc_miou_results.json',
    OUTPUT_ROOT / f'to_epoch_{FINAL_EPOCH:04d}' / 'voc_all_checkpoints' / 'voc_miou_results.json',
    Path('/kaggle/input/dino-dense-degradation-eval') / f'to_epoch_{FINAL_EPOCH:04d}' / 'voc_all_checkpoints' / 'voc_miou_results.json',
    Path('/kaggle/input/datasets/bingzhouxie1/dino-dense-degradation-eval') / f'to_epoch_{FINAL_EPOCH:04d}' / 'voc_all_checkpoints' / 'voc_miou_results.json',
]
for root in [OUTPUT_ROOT, Path('/kaggle/input/dino-dense-degradation-eval'), Path('/kaggle/input/datasets/bingzhouxie1/dino-dense-degradation-eval')]:
    if root.is_dir():
        VOC_JSON_CANDIDATES.extend(sorted(root.glob('to_epoch_*/voc_all_checkpoints/voc_miou_results.json'), reverse=True))
VOC_JSON_FOR_REPORT = next((path for path in VOC_JSON_CANDIDATES if path.is_file()), None)

print('prepared epochs:', SELECTED_EPOCHS)
print('final epoch:', FINAL_EPOCH)
print('run output root:', RUN_OUTPUT_ROOT)
print('VOC json for report:', VOC_JSON_FOR_REPORT)


## 4. Confirm VOC handling

This Kaggle notebook does not train VOC heads by default. If a previous VOC JSON is attached as a Kaggle input, plots/reports will include it. Otherwise plots/reports are generated without VOC.


In [ ]:
if RUN_VOC_EVAL:
    raise NotImplementedError(
        'This Kaggle raw/L2 notebook is patch-diagnostics only. '
        'Use the Colab notebook or add a Kaggle VOC evaluator cell if VOC must be rerun.'
    )

if VOC_JSON_FOR_REPORT is None:
    print('No VOC JSON found. Continuing with patch-only raw/L2 diagnostics.')
else:
    print('Using VOC JSON:', VOC_JSON_FOR_REPORT)


## 5. Run raw/L2 patch diagnostics

This is the main evaluation step. For the smoke test, it runs only epoch 215 with fewer sampled images. For full evaluation, change the configuration in section 2.


In [ ]:
cmd = [
    sys.executable,
    str(REPO_DIR / 'analyze_patch_statistics.py'),
    '--ckpt_dir', str(WORK_CKPT_DIR),
    '--image_root', str(DSE_IMAGE_ROOT),
    '--out', str(RUN_OUTPUT_ROOT / 'patch_attention_dse_all_checkpoints'),
    '--arch', 'vit_small',
    '--patch_size', '16',
    '--checkpoint_key', CHECKPOINT_KEY,
    '--num_metric_images', str(NUM_DSE_IMAGES),
    '--num_vis_images', str(NUM_VIS_IMAGES),
    '--batch_size', str(PATCH_DIAG_BATCH_SIZE),
    '--num_workers', str(PATCH_DIAG_NUM_WORKERS),
    '--max_spectrum_tokens', str(MAX_SPECTRUM_TOKENS),
    '--max_kmeans_tokens', str(MAX_KMEANS_TOKENS),
    '--dse_group_stride', str(PATCH_DSE_GROUP_STRIDE),
    '--seed', '0',
]
if STRICT_INTERNAL_EPOCH:
    cmd.append('--strict_internal_epoch')

print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 6. Plot curves and write report


In [ ]:
summary_csv = RUN_OUTPUT_ROOT / 'patch_attention_dse_all_checkpoints' / 'patch_attention_dse_summary.csv'
fig_dir = RUN_OUTPUT_ROOT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

plot_cmd = [
    sys.executable,
    str(REPO_DIR / 'plot_dense_diagnostics.py'),
    '--summary_csv', str(summary_csv),
    '--out_dir', str(fig_dir),
]
if VOC_JSON_FOR_REPORT is not None:
    plot_cmd.extend(['--voc_json', str(VOC_JSON_FOR_REPORT)])
print(' '.join(plot_cmd))
subprocess.run(plot_cmd, check=True)

report_cmd = [
    sys.executable,
    str(REPO_DIR / 'make_summary_report.py'),
    '--summary_csv', str(summary_csv),
    '--out', str(RUN_OUTPUT_ROOT / 'summary_report.md'),
]
if VOC_JSON_FOR_REPORT is not None:
    report_cmd.extend(['--voc_json', str(VOC_JSON_FOR_REPORT)])
print(' '.join(report_cmd))
subprocess.run(report_cmd, check=True)


## 7. Inspect outputs


In [ ]:
import pandas as pd

print('RUN_OUTPUT_ROOT:', RUN_OUTPUT_ROOT)
print('summary exists:', summary_csv.is_file(), summary_csv)
print('report exists:', (RUN_OUTPUT_ROOT / 'summary_report.md').is_file())

for name in [
    'fig_dense_diagnostics_summary.png',
    'fig_raw_vs_l2_dse.png',
    'fig_raw_vs_l2_class_sep.png',
    'fig_raw_vs_l2_spectrum.png',
    'combined_dense_summary.csv',
]:
    path = fig_dir / name
    print(name, path.is_file(), path)

df = pd.read_csv(summary_csv)
required = [
    'raw_dse',
    'l2_dse',
    'raw_class_sep_avg',
    'l2_class_sep_avg',
    'raw_effective_rank',
    'l2_effective_rank',
    'raw_top1_eigen_ratio',
    'l2_top1_eigen_ratio',
    'patch_norm_mean',
    'patch_norm_p90',
]
for col in required:
    print(col, col in df.columns)

display_cols = ['epoch'] + [col for col in required if col in df.columns]
display(df[display_cols])

print('\n=== summary_report.md ===')
print((RUN_OUTPUT_ROOT / 'summary_report.md').read_text()[:5000])


## 8. Archive outputs for Kaggle download

Kaggle automatically saves `/kaggle/working`, but a tarball makes downloading the exact run easier.


In [ ]:
import tarfile

archive_path = Path('/kaggle/working') / f'{RUN_OUTPUT_ROOT.name}.tar.gz'
if archive_path.exists():
    archive_path.unlink()
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(RUN_OUTPUT_ROOT, arcname=RUN_OUTPUT_ROOT.name)
print('archive:', archive_path, archive_path.stat().st_size / 1024 / 1024, 'MB')
